# 3D GPU nucleation of Brownian particles (CUDA)

Free particles random-walk in a cubic voxel box, hopping ±x, ±y or ±z each step (walls reflect by staying put). One immobile nucleation seed sits at the centre. The simulation runs on the GPU via `numba.cuda`.

**Size (viewer, applied with a reset):** the box side (8 to 1,000 voxels, default 200) and the number of particles (1 to 1,000,000, default 10,000). One compiled kernel serves every size.

Rules:
* **Cluster formation:** whenever any 2x2x2 window holds 3 or more particles, they are bound to each other into one cluster (merging any clusters they already belong to). A checkbox in the viewer turns this rule off; then particles only bind by touching the seed's cluster or another cluster, as in classic diffusion-limited aggregation.
* **Clusters keep diffusing:** a cluster moves as one rigid body, one voxel in a random direction with probability 1/size per step (a free particle always hops). If any member would hit a wall, the cluster stays put that step.
* **Sticking:** a cluster sticks to anything it touches (same or face-adjacent voxel): free particles and other clusters join it. Free particles touching each other do not stick (only the 2x2x2 rule binds them).
* **Seed:** anything that touches the seed's cluster joins it and stops moving. The **Anchor** button removes or restores this central anchor cell: removing it sets the system stuck to it floating (regrouped by contact into pieces that start at rest); restoring it puts the anchor back at the centre, where anything touching it sticks.
* **Mass ceiling and ejection:** a floating mass (mobile cluster) at or above the ceiling (slider, default 200 particles; 'off' at the top) ejects a piece of itself as a **first-order process**: every step it fires with probability *k* (eject rate slider, default 0.01/step; 1 = as soon as it reaches the ceiling), so its waiting time is exponential with mean 1/*k* steps. The piece is a set percentage of the mass (ejecta slider, default 25%; at least 1 particle, and at least 1 left behind), taken from the side facing the ejection direction and shot off that way. The rest of the mass recoils the opposite way. Momentum is conserved, so the lighter part moves faster (relative speed: eject speed slider, in voxels per step). The two can't re-stick to each other until they have had time to separate. Every mass that fires in a step ejects that step, one after another.
* **First-order decay (button, off by default):** the ceiling no longer gates ejection: every floating mass of 2 or more particles fires with probability *k* per step, so each one sheds mass at a rate proportional to its mass and decays exponentially (dm/dt = -k · ejecta fraction · m, until it breaks up). The ceiling still sets which masses get coloured.
* **Diffusers and aggregators (slider, default 50% aggregators):** a **diffuser** ejects into empty space, directly away from the nearest other floating mass. An **aggregator** ejects directly towards the nearest other floating mass, feeding it. Both pick a random direction if there is no other floating mass. Each floating mass is an aggregator with the probability set by the slider. The draw is fixed per mass: a hash of its root particle, the lowest-indexed particle in it. Moving the slider therefore converts existing masses straight away, and only in one direction (raising it turns diffusers into aggregators). A mass formed by a merge or an ejection has a new root, so its kind is drawn afresh.
* **Colours (K of them, slider, default 3):** every floating mass has a colour, which changes only two ways: a mass that attaches to a larger mass takes on that mass's colour (when several merge at once, the biggest one's colour wins), and a new mass formed by free particles aggregating takes a random colour. Ejected pieces keep their parent's colour.
* **Colour rule (button, off by default):** replaces diffusers and aggregators. A mass ejects directly towards the nearest other floating mass if that mass has a different colour, and directly away from it if it has the same colour. Its rate of ejection is inversely proportional to the distance between the two masses' centres: rate = *k* · *d₀* / *d* (the distance slider sets *d₀*, default 20, the distance at which the rate is the eject rate *k*; *d* is at least 1, and with no other floating mass it is the box diagonal).
* **Evolution (button, off by default):** each of the K colours carries its own random set of features: eject rate, ejecta fraction and eject speed (each between ¼ and 4 times its slider's value), and aggregator chance (0 to 100%, the probability that a mass of that colour is an aggregator). A mass uses its colour's features. Each mass's lifetime is measured from when its root first becomes a floating mass until that root stops being one: it merges into another mass, joins the seed, or breaks down to a single particle. After an ejection, the root's own piece carries on. Each colour keeps a running mean of its masses' lifetimes (its survival). Every time a new mass of a colour appears, the colour **mutates** with probability *μ* · exp(−survival / *τ*) (sliders: mutation chance *μ*, default 20%, and survival scale *τ*, default 1,000 steps), so a colour whose masses survive long rarely changes. A mutation redraws each feature with that feature's **mutation weight** (one slider per feature, default 50%; 0 = never changes). A table lists each colour's live masses, features, mean survival and mutations. With the colour rule also on, it sets the eject direction and the aggregator chance goes unused.
* **Momentum:** moving masses keep their Brownian jitter while drifting, bounce off the walls by the law of reflection (the velocity component normal to the wall flips), and slow down gradually (damping slider). Masses that merge share their momentum; anything that joins the seed's system stops.

Colours (ochre background): free particles are faint white dots. Bound particles (in any cluster) are shaded by the step at which they bound, from **black** (seed / bound first) to **white** (bound most recently).

**Masses near the ceiling (button, on by default, and two sliders):** while the button and the ceiling are on, every floating mass at or above a set fraction of the ceiling (default 80%) is drawn in a colour of its own: its colour-rule colour while the colour rule or evolution is on, otherwise a random colour from its root particle, which stays the same while the mass does. Every other cell (free particles, smaller masses, the anchored system) is drawn at a set opacity (default 30%).

**Blinks (button, on by default):** when a floating particle or cluster attaches to the seed's cluster, every particle already in the seed's cluster turns blue for a moment and back, and then the arriving system's particles do the same after a delay inversely proportional to its size (delay = base delay / number of particles; the base delay for a single particle is a slider, default 1 s).

View: z is up. **Drag** to rotate, **scroll** to zoom, **double-click** to reset the view. The viewer is a small web page served from the kernel and shown in the cell output (the printed link opens it in a browser too); it is drawn in the browser, so rotating is smooth even while the simulation runs.

How it works on the GPU (~2-3.5k steps/s for the default 10,000 particles on an MX330; ~10 steps/s for a million):
* Clusters are tracked with a lock-free **union-find** (each particle has a parent pointer; merges link roots with an atomic compare-and-swap). After a merge, only the roots that got linked pass their size and "stuck to seed" flag on to their new root.
* Neighbours are found with a **cell list** rebuilt every step: particles are bucketed into cells of 8x8x8 voxels (16x16x16 for boxes over 512, so there are at most ~64 cells a side), with one atomic per particle; each warp reserves space for 32 cells with a single atomic, and positions are copied into cell order so a particle reads its neighbours' positions directly.
* Only particles that moved, or whose cluster status just changed, look for neighbours: any new contact or newly full 2x2x2 window must involve one of them. A quick first pass skips particles with nothing from another cluster within one voxel, which is almost all of them.
* The phases of a step depend on each other (choose cluster moves and check walls, move, build the cell list, interact, merge, eject), so the kernel runs on the whole GPU as a **cooperative launch** with grid-wide syncs between phases. Each launch is timed and the number of steps per launch adapts so it takes ~0.1 s, well under the display driver's watchdog at any size.
* Random numbers come from a cheap stateless hash of (particle, step). Every member of a cluster hashes the root's index, so they all agree on the cluster's move without talking to each other; the move is stored once and reused by the next phase.
* Ejections find the nearest other mass with a minimum taken across each warp first, so only one atomic per warp hits the shared result.
* Each frame the GPU packs every particle into 12 bytes (position, shade and flags, attach mass or, for a mass near the ceiling, its root). A frame is only packed once the browser has drawn the last one, so a big system doesn't copy frames nobody sees, and nothing is resent while paused. The browser draws with a depth buffer instead of sorting, so a million particles stay interactive, and spots newly attached particles by comparing frames to time the blinks.

Run the cells in order. No extra packages beyond `numba` and `numpy`.


In [ ]:
import json, threading, time, warnings
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import parse_qs, urlparse
from IPython.display import HTML, IFrame, display
from numba import cuda

warnings.filterwarnings('ignore', category=Warning, module='numba')
print(cuda.get_current_device().name.decode())

In [ ]:
import math
import numpy as np
from numba import cuda, int32, int64, uint32

LMAX = 1000               # largest box side in voxels (a position packs 3 x PB bits)
NMAX = 1_000_000          # most free particles
PB = 10                   # bits per coordinate: a position is x | y<<PB | z<<2PB
PM = (1 << PB) - 1
IB = 20                   # bits for a particle index packed under a squared distance (G2)
WIN = 3                   # a 2x2x2 window holding >= WIN particles fuses them into one cluster
TPB = 256
LAUNCH = 0.1              # target seconds per kernel launch: steps per launch adapt to stay well under the display watchdog
BIG = 1 << 62
HB = 4096                 # ejection: histogram bins for the projection of a mass onto the eject direction
OFF = 1 << 30             # mass ceiling 'off'
KMAX = 8                  # most colours
NF = 4                    # evolution: features per colour (eject rate, ejecta fraction, eject speed, aggregator chance)
SPREAD = 4.0              # evolution: a colour's rate, ejecta fraction and speed lie in [1/SPREAD, SPREAD] x the slider
EMA = 0.1                 # evolution: weight of each new lifetime in a colour's running mean survival
INV = 1.0 / 4294967296.0  # uint32 hash -> [0, 1)
C1, C2, C3, C4 = 0x9E3779B1, 0x85EBCA77, 0x2C1B3C6D, 0x297A2D39   # hash multipliers
assert LMAX <= 1 << PB and NMAX + 1 < 1 << IB


def geometry(L):
    """Cell list for a box of side L: (2^CS)^3-voxel cells, at least 8 voxels and at most ~64 cells a side."""
    CS = max(3, ((L - 1) // 64).bit_length())
    NCC = (L + (1 << CS) - 1) >> CS
    return CS, NCC, NCC ** 3


@cuda.jit(device=True)
def rnd(i, t):
    """Stateless hash of (particle, step) -> uint32."""
    h = uint32(uint32(i) * uint32(C1) + uint32(uint32(t) * uint32(C2)))
    h ^= h >> uint32(15)
    h = uint32(h * uint32(C3))
    h ^= h >> uint32(12)
    h = uint32(h * uint32(C4))
    h ^= h >> uint32(16)
    return h


@cuda.jit(device=True)
def decide(r, t, sz):
    """Move of the cluster with root r at step t: direction 0..5 (+x -x +y -y +z -z), or -1 to stay.
    Every member computes the same answer from the root's index, so no extra phase is needed."""
    h = rnd(r, t)
    if (h & uint32(0xFFFF)) * uint32(sz) >= uint32(0x10000):          # hop with probability 1/size
        return -1
    return int32((np.uint64(h) * np.uint64(6)) >> np.uint64(32))


@cuda.jit(device=True)
def drift(r, sz, mom, acc):
    """Ballistic part of a cluster's motion. Its velocity mom[r]/size is added to a sub-voxel
    accumulator; when an axis has built up a whole voxel, the cluster hops along that axis this step.
    Returns the updated accumulator and that axis (0..2, or -1 for none)."""
    inv = 1.0 / sz
    a0 = acc[r, 0] + mom[r, 0] * inv
    a1 = acc[r, 1] + mom[r, 1] * inv
    a2 = acc[r, 2] + mom[r, 2] * inv
    ax = -1
    best = 1.0
    if math.fabs(a0) >= best:
        ax, best = 0, math.fabs(a0)
    if math.fabs(a1) >= best:
        ax, best = 1, math.fabs(a1)
    if math.fabs(a2) >= best:
        ax, best = 2, math.fabs(a2)
    return a0, a1, a2, ax


@cuda.jit(device=True)
def move_dir(r, t, sz, mom, acc):
    """This step's hop for the cluster with root r: a ballistic hop if one is due, else a Brownian one."""
    a0, a1, a2, ax = drift(r, sz, mom, acc)
    if ax == 0:
        return 0 if a0 > 0 else 1
    if ax == 1:
        return 2 if a1 > 0 else 3
    if ax == 2:
        return 4 if a2 > 0 else 5
    return decide(r, t, sz)


@cuda.jit(device=True)
def cell_of(p, CS, NCC):
    """Cell-list index of packed position p."""
    return ((int32(p >> (2 * PB)) >> CS) * NCC + (int32((p >> PB) & PM) >> CS)) * NCC + (int32(p & PM) >> CS)


@cuda.jit(device=True)
def is_agg(r, pagg, NP):
    """Is the floating mass with root r an aggregator? A fixed hash of the root against the probability pagg,
    so moving the slider converts masses right away (raising it only turns diffusers into aggregators)."""
    return rnd(r + 7 * NP, 0) * (1.0 / 4294967296.0) < pagg


@cuda.jit(device=True)
def colour(col, r, ncol):
    """Colour 0..ncol-1 of the mass with root r (colour rule, evolution). Colours are stored per root and
    change only two ways: a merge takes the colour of the biggest mass in it, and a mass formed from free
    particles alone draws a random one (F). Ejected pieces keep their parent's colour (G7)."""
    return int32(col[r]) % ncol


@cuda.jit(device=True)
def clampi(v, lo, hi):    # (no min/max: this numba's CUDA target can't compile them)
    return lo if v < lo else (hi if v > hi else v)


@cuda.jit(device=True)
def scaled(base, u):
    """Evolution: a colour's value of a slider-based feature, from its feature u in [0, 1)."""
    return base * math.pow(SPREAD, 2.0 * u - 1.0)


@cuda.jit(device=True)
def find(par, i):
    while par[i] != i:
        i = par[i]
    return i


@cuda.jit(device=True)
def union(par, a, b):
    """Lock-free union-find link: the larger root points at the smaller one (atomic CAS)."""
    while True:
        a = find(par, a)
        b = find(par, b)
        if a == b:
            return
        if a < b:
            a, b = b, a
        if cuda.atomic.cas(par, a, a, b) == a:
            return


@cuda.jit(max_registers=128)        # caps register use so 2 blocks fit per SM (~1.5x faster than uncapped)
def advance(pos, par, size, fixed, clus, btime, blocked, act, mv, froot, key, slot, ccount, cstart, order, spos,
            sroot, slot2, ccount2, cstart2, order2, spos2, sroot2,
            osize, amass, mom, acc, gid, gtil, misc, hist, ejc, fire, flag, stats,
            born, feat, surv, lsum, lcnt, nmut, evp, col, cbest, csum, srcs, cand,
            L, N, CS, t0, steps, win, cap, v0, damp, grace, anchor, erate, efrac, pagg, crule, ncol, evo, dref):
    """Advance the whole system `steps` steps; phases are separated by grid-wide syncs.

    The box side L, the particle count N and the cell size 2^CS are arguments, so one compiled kernel
    serves every size (NP = N + 1 with the seed, NCC cells a side, NCELL cells).
      A. each member of a mobile cluster computes its cluster's move (stored in mv for B) and flags the
         cluster if it would hit a wall. The move is ballistic if the cluster's momentum has built up a
         whole voxel of drift on some axis, else Brownian (hop with probability 1/size); both are decided
         from the root's state and index, so all members agree. Also marks the floating masses at or
         above the ceiling that eject this step: each one fires with probability `erate` per step (a
         first-order process; with first-order decay the host passes cap = 2, so every mass can fire)
      B. unblocked clusters move rigidly; every particle takes a slot in its cell (cell list, part 1).
         Two cell lists are built: one of all particles, and one of clustered particles only
      C. each cell gets a contiguous range of the sorted array (part 2). Cells don't need to be in
         order, so instead of a slow serial prefix sum each warp scans 32 cell counts with shuffles
         and reserves space for all of them with a single atomic on a global cursor
      D. particles (index and position) are scattered into cell order (part 3); cluster roots
         update their drift accumulator, reflect their momentum off a wall they hit (law of
         reflection: the normal component flips), then damp it
      D2. colour rule only (crule): the firing is decided here instead of in A, at a rate inversely
         proportional to the distance d from the mass's centroid to the nearest other floating mass's
         centroid: rate = k * dref / d (d at least 1; with no other mass, d = the box diagonal).
         Centroids are summed in B and the floating masses and the candidates (size >= cap) listed in C;
         a warp takes each candidate, its 32 lanes splitting the list
      E. interactions, only for particles that moved or whose cluster status just changed (a new
         contact or a newly full 2x2x2 window always involves one of them); free particles always
         check. Each one looks at the particles in the cells around it and builds its 3x3x3
         neighbourhood. A free particle with the 2x2x2 rule on looks at everyone; otherwise only
         clustered particles matter (a free particle can only stick to a cluster, and any contact
         with a free particle is found by that free particle), so it scans the much shorter
         clustered-only list. A quick pass first stops at the first particle of another cluster
         within one voxel, so particles touching only their own cluster cost little:
         - any 2x2x2 window around it with >= WIN particles: union everything in it (only if win)
         - contact (same or face-adjacent voxel) with a different cluster, when either side is a
           cluster (size >= 2) or the immobile seed cluster: union
      F. only if something merged: note each particle's pre-merge cluster size, path-compress, and
         each root that got linked this step adds its
         size (and 'fixed' flag) to its new root, so only merged clusters are touched; then stamp
         bind times and mark newly clustered particles to be checked next step. Momentum adds up
         on merging; a system joining the seed's cluster stops. Colour: each old root that got linked
         bids its pre-merge size and colour, and so does its new root (cbest, an atomic max of
         size * 16 + colour); the merged mass takes the biggest one's colour, or a random one if they
         were all single particles (a new mass formed from free particles).
      G. ejections: every mass marked in A, one after another, lowest root first. Find its centroid
         and the nearest particle of another floating mass, eject a fraction `efrac` of its particles
         (at least 1, leaving at least 1) from the side facing the eject direction, and give the two
         parts equal and opposite momentum kicks (relative speed v0), so the lighter part moves faster.
         The direction: with the colour rule (crule), away from that mass if it has the same colour
         (colour(root, ncol)), towards it if not; otherwise away from it for a diffuser, towards it for
         an aggregator (is_agg(root, pagg)). Each part is then regrouped by contact (a part that isn't
         in one piece breaks into fragments that share its velocity), and the pieces can't re-stick to
         each other for `grace` steps.
      H. evolution (only if evo): a mass is born when its root first becomes a floating mass (size >= 2)
         and ends when that root stops being one (merged into another root, joined the seed, or broken
         down to one particle; after an ejection the root's own piece carries on). An ending adds its
         lifetime to its colour's tally (lsum, lcnt), folded into the colour's running mean survival
         surv[c] at the start of the next step. A birth mutates its colour with probability
         evp[0] * exp(-surv[c] / evp[1]); a mutation redraws each feature f with probability evp[2 + f].
         With evo on, a mass's eject rate, ejecta fraction and eject speed are scaled(slider, feat[c, f])
         and its aggregator chance is feat[c, 3] (c = colour(root, ncol)).
    size[r] is only meaningful for roots (it is zeroed when a root gets linked).
    amass[i]: when particle i joined the seed's cluster, the mass (particle count) of the floating
    system it arrived in (0 = not attached; the viewer uses it for the blink delay).
    flag: [merged, cell cursor, clustered-list cursor, D2 list lengths x2].
    mom[r], acc[r]: root's momentum and drift accumulator.  fire[r] = t: mass r ejects at step t.
    col[r]: colour of the mass with root r.  cbest[r]: F's merge bids (-1 = none).  csum[r]: D2's position sums.
    srcs, cand: D2's lists of floating masses and of the ones that may fire (lengths flag[3], flag[4]).
    gid/gtil: ejection grace (roots with the same gid don't stick before step gtil).
    misc: [-, nearest, ejecta size, centroid sums x3, threshold bin, need, tie count, -, -,
    highest histogram bin, next mass to eject x2 (by step parity, so a fast thread starting the next
    step can't change what a slow one is about to read)].
    ejc: part label during an ejection (1 ejecta, 2 rest).  stats: [last bind step, bound, fixed, ejections].
    anchor = 0: the central anchor (particle N) is switched off: it is skipped by every interaction
    (Sim.set_anchor detaches its system on the host).
    pos: packed x | y<<PB | z<<2PB.  par: union-find parent (par[i] = root outside phases E/F).
    Cell lists: ccount/cstart/order/spos/sroot (all particles; sroot = cluster root at the start of
    the step), and the same with a 2 for clustered particles only.
    clus[i]: particle is in a cluster (size >= 2) or fixed.
    """
    grid = cuda.cg.this_grid()
    g0 = cuda.grid(1)
    gs = cuda.gridsize(1)
    nbh = cuda.local.array(27, int32)
    lane = cuda.threadIdx.x & 31
    NP = N + 1
    NCC = (L + (1 << CS) - 1) >> CS
    NCELL = NCC * NCC * NCC
    hr = 64.0 if L <= 256 else L / 4.0                                 # ejection histogram: +-hr voxels
    hsc = HB / (2.0 * hr)                                              # (bins per voxel)
    for k in range(steps):
        t = t0 + k + 1
        es = 12 + (t & 1)                                              # misc slot: next mass to eject
        for c in range(g0, NCELL, gs):
            ccount[c] = 0
            ccount2[c] = 0
        if g0 == 0:
            flag[1] = 0                                                # cell-range allocation cursors
            flag[2] = 0
            flag[3] = 0                                                # D2's list lengths
            flag[4] = 0
        for i in range(g0, NP, gs):                                    # A. moves + wall check
            if fixed[i] == 0:
                r = par[i]
                if r == i and size[i] >= cap and crule == 0:           # (colour rule: decided in D2)
                    k1 = scaled(erate, feat[colour(col, i, ncol), 0]) if evo else erate
                    if rnd(i + 6 * NP, t) * INV < k1:
                        fire[i] = t                                    # ejects this step (first order)
                        cuda.atomic.min(misc, es, i)
                d = move_dir(r, t, size[r], mom, acc)
                mv[i] = d
                if d >= 0:
                    c = int32(pos[i] >> uint32((d >> 1) * PB)) & PM
                    if (d & 1 == 0 and c == L - 1) or (d & 1 == 1 and c == 0):
                        blocked[r] = t
        grid.sync()
        if g0 == 0:
            misc[12 + ((t + 1) & 1)] = NP                              # next step's slot (read 2 steps ago)
            if evo:                                                    # H (from the last step): fold the
                for c in range(KMAX):                                  # lifetimes that ended into each
                    nd = lcnt[c]                                       # colour's running mean survival
                    if nd > 0:
                        surv[c] += (1.0 - math.pow(1.0 - EMA, nd)) * (lsum[c] / nd - surv[c])
                        lsum[c] = 0.0
                        lcnt[c] = 0
        for i in range(g0, NP, gs):                                    # B. rigid moves + cell counts
            a = act[i] == 2                                            # 2 = re-check requested by F
            p = pos[i]
            if fixed[i] == 0:
                d = mv[i]                                              # (A's move: not worked out again)
                if d >= 0 and blocked[par[i]] != t:
                    s = uint32(1) << uint32((d >> 1) * PB)
                    p = p + s if d & 1 == 0 else p - s
                    pos[i] = p
                    a = True
            act[i] = 1 if a or clus[i] == 0 else 0                    # free particles always check
            c = cell_of(p, CS, NCC)
            key[i] = c
            slot[i] = cuda.atomic.add(ccount, c, 1)
            if clus[i]:
                slot2[i] = cuda.atomic.add(ccount2, c, 1)
                if crule and fixed[i] == 0 and size[par[i]] >= 2:     # D2: centroid sums of floating masses
                    r = par[i]
                    cuda.atomic.add(csum, (r, 0), int64(p & PM))
                    cuda.atomic.add(csum, (r, 1), int64((p >> PB) & PM))
                    cuda.atomic.add(csum, (r, 2), int64(p >> (2 * PB)))
        grid.sync()
        for it in range((NCELL + gs - 1) // gs):                       # C. allocate cell ranges
            c = g0 + it * gs                                           # (whole warps take part)
            n = ccount[c] if c < NCELL else 0
            v = n                                                      # inclusive warp scan
            for d in (1, 2, 4, 8, 16):
                u = cuda.shfl_up_sync(0xFFFFFFFF, v, d)
                if lane >= d:
                    v += u
            base = 0
            if lane == 31:
                base = cuda.atomic.add(flag, 1, v)
            base = cuda.shfl_sync(0xFFFFFFFF, base, 31)
            if c < NCELL:
                cstart[c] = base + v - n
            n = ccount2[c] if c < NCELL else 0                         # and for the clustered-only list
            v = n
            for d in (1, 2, 4, 8, 16):
                u = cuda.shfl_up_sync(0xFFFFFFFF, v, d)
                if lane >= d:
                    v += u
            base = 0
            if lane == 31:
                base = cuda.atomic.add(flag, 2, v)
            base = cuda.shfl_sync(0xFFFFFFFF, base, 31)
            if c < NCELL:
                cstart2[c] = base + v - n
        if crule:                                                      # D2's lists: floating masses, and those
            for i in range(g0, NP, gs):                                # big enough to fire
                if par[i] == i and fixed[i] == 0 and size[i] >= 2:
                    srcs[cuda.atomic.add(flag, 3, 1)] = i
                    if size[i] >= cap:
                        cand[cuda.atomic.add(flag, 4, 1)] = i
        grid.sync()
        for i in range(g0, NP, gs):                                    # D. scatter into cell order
            q = cstart[key[i]] + slot[i]
            order[q] = i
            spos[q] = pos[i]
            sroot[q] = par[i]
            if clus[i]:
                q = cstart2[key[i]] + slot2[i]
                order2[q] = i
                spos2[q] = pos[i]
                sroot2[q] = par[i]
            if fixed[i] == 0 and par[i] == i:                          # root: drift, wall reflection, damping
                a0, a1, a2, ax = drift(i, size[i], mom, acc)
                if ax >= 0:                                            # this step's hop was ballistic
                    a = a0 if ax == 0 else (a1 if ax == 1 else a2)
                    if blocked[i] == t:                                # hit a wall: reflect
                        mom[i, ax] = -mom[i, ax]
                        a = 0.0
                    else:
                        a -= 1.0 if a > 0 else -1.0
                    if ax == 0:
                        a0 = a
                    elif ax == 1:
                        a1 = a
                    else:
                        a2 = a
                acc[i, 0] = a0
                acc[i, 1] = a1
                acc[i, 2] = a2
                sz = float(size[i])
                vl1 = (math.fabs(mom[i, 0]) + math.fabs(mom[i, 1]) + math.fabs(mom[i, 2])) / sz
                for c3 in range(3):                                    # damping; and the lattice can't go
                    mom[i, c3] *= (1.0 - damp) / (vl1 if vl1 > 1.0 else 1.0)   # faster than 1 hop/step
        grid.sync()
        if crule:                                                      # D2. colour rule: rate ~ 1/distance
            nm = flag[3]
            for w in range(g0 >> 5, flag[4], gs >> 5):                 # a warp per candidate mass; its lanes
                i = cand[w]                                            # split the list of floating masses
                m = float(size[i])
                cx = csum[i, 0] / m
                cy = csum[i, 1] / m
                cz = csum[i, 2] / m
                best = 1e30                                            # nearest other centroid
                for q in range(lane, nm, 32):
                    r2 = srcs[q]
                    if r2 != i:
                        m2 = float(size[r2])
                        ux = csum[r2, 0] / m2 - cx
                        uy = csum[r2, 1] / m2 - cy
                        uz = csum[r2, 2] / m2 - cz
                        d2 = ux * ux + uy * uy + uz * uz
                        if d2 < best:
                            best = d2
                for d in (16, 8, 4, 2, 1):
                    u = cuda.shfl_xor_sync(0xFFFFFFFF, best, d)
                    if u < best:
                        best = u
                if lane == 0:
                    dist = math.sqrt(best) if best < 1e29 else 1.7320508 * L
                    k1 = scaled(erate, feat[colour(col, i, ncol), 0]) if evo else erate
                    k1 *= dref / (dist if dist > 1.0 else 1.0)
                    if rnd(i + 6 * NP, t) * INV < k1:
                        fire[i] = t                                    # ejects this step
                        cuda.atomic.min(misc, es, i)
            grid.sync()                                                # (everyone has read the sums; and E's
            for q in range(g0, nm, gs):                                #  unions relink roots D2 reads)
                csum[srcs[q], 0] = 0
                csum[srcs[q], 1] = 0
                csum[srcs[q], 2] = 0
        for i in range(g0, NP, gs):                                    # E. interactions
            if act[i] == 0 or (i == N and anchor == 0):
                continue
            p = pos[i]
            x = int32(p & PM)
            y = int32((p >> PB) & PM)
            z = int32(p >> (2 * PB))
            cz0 = (z - 1 if z > 0 else 0) >> CS                         # (no min/max: this numba's CUDA
            cz1 = (z + 1 if z < L - 1 else z) >> CS                     #  target can't compile them)
            cy0 = (y - 1 if y > 0 else 0) >> CS
            cy1 = (y + 1 if y < L - 1 else y) >> CS
            cx0 = (x - 1 if x > 0 else 0) >> CS
            cx1 = (x + 1 if x < L - 1 else x) >> CS
            if clus[i] == 0 and win == 1:                              # free, 2x2x2 rule on: everyone
                cs, cc, od, sp, sr = cstart, ccount, order, spos, sroot
            else:                                                      # otherwise: clustered ones only
                cs, cc, od, sp, sr = cstart2, ccount2, order2, spos2, sroot2
            ri = par[i]
            found = False
            for cz in range(cz0, cz1 + 1):                             # pass 0: another cluster within
                for cy in range(cy0, cy1 + 1):                         # one voxel? (stops at the first)
                    for cx in range(cx0, cx1 + 1):
                        c = (cz * NCC + cy) * NCC + cx
                        q0 = cs[c]
                        for q in range(q0, q0 + cc[c]):
                            pj = sp[q]
                            dx = int32(pj & PM) - x
                            dy = int32((pj >> PB) & PM) - y
                            dz = int32(pj >> (2 * PB)) - z
                            if -1 <= dx <= 1 and -1 <= dy <= 1 and -1 <= dz <= 1:
                                if sr[q] != ri and (anchor or od[q] != N):
                                    found = True
                                    break
                        if found:
                            break
                    if found:
                        break
                if found:
                    break
            if not found:                                              # alone, or only touching its own
                continue                                               # cluster: nothing new can happen
            wmask = 0                                                  # which of the 8 windows are full
            if win:
                for o in range(27):
                    nbh[o] = 0
                for cz in range(cz0, cz1 + 1):                         # pass 1: neighbourhood counts
                    for cy in range(cy0, cy1 + 1):
                        for cx in range(cx0, cx1 + 1):
                            c = (cz * NCC + cy) * NCC + cx
                            q0 = cs[c]
                            for q in range(q0, q0 + cc[c]):
                                pj = sp[q]
                                dx = int32(pj & PM) - x
                                dy = int32((pj >> PB) & PM) - y
                                dz = int32(pj >> (2 * PB)) - z
                                if -1 <= dx <= 1 and -1 <= dy <= 1 and -1 <= dz <= 1 and (anchor or od[q] != N):
                                    nbh[((dz + 1) * 3 + dy + 1) * 3 + dx + 1] += 1
                for w in range(8):
                    wx = w & 1
                    wy = (w >> 1) & 1
                    wz = w >> 2
                    s = 0
                    for oz in range(2):
                        for oy in range(2):
                            for ox in range(2):
                                s += nbh[((wz + oz) * 3 + wy + oy) * 3 + wx + ox]
                    if s >= WIN:
                        wmask |= 1 << w
            gi = gid[ri]                                               # ejection grace of i's mass
            gt = gtil[ri]
            for cz in range(cz0, cz1 + 1):                             # pass 2: unions
                for cy in range(cy0, cy1 + 1):
                    for cx in range(cx0, cx1 + 1):
                        c = (cz * NCC + cy) * NCC + cx
                        q0 = cs[c]
                        for q in range(q0, q0 + cc[c]):
                            pj = sp[q]
                            dx = int32(pj & PM) - x
                            dy = int32((pj >> PB) & PM) - y
                            dz = int32(pj >> (2 * PB)) - z
                            if not (-1 <= dx <= 1 and -1 <= dy <= 1 and -1 <= dz <= 1):
                                continue
                            j = od[q]
                            if j == i or (j == N and anchor == 0) or sr[q] == ri:
                                continue
                            join = False
                            if wmask:
                                for w in range(8):                     # j inside a full window?
                                    if wmask & (1 << w):
                                        wx = (w & 1) - 1
                                        wy = ((w >> 1) & 1) - 1
                                        wz = (w >> 2) - 1
                                        if wx <= dx <= wx + 1 and wy <= dy <= wy + 1 and wz <= dz <= wz + 1:
                                            join = True
                            if dx * dx + dy * dy + dz * dz <= 1 and (clus[i] or clus[j]):
                                join = True                            # contact with a cluster / the seed
                            if join and gi != 0 and gi == gid[sr[q]] and t < gt:
                                join = False                           # ejecta and its parent: grace period
                            if join and find(par, i) != find(par, j):
                                union(par, i, j)
                                flag[0] = 1
        grid.sync()
        if flag[0]:                                                    # F. only after merges
            for i in range(g0, NP, gs):                                # pre-merge size of i's cluster
                osize[i] = size[i] if size[i] > 0 else size[par[i]]    # (size > 0 marks an old root)
                if size[i] > 0:
                    r = find(par, i)
                    if r != i:                                         # an old root that got linked: it and
                        cuda.atomic.max(cbest, r, int64(size[i]) * 16 + col[i])   # its new root bid
                        cuda.atomic.max(cbest, r, int64(size[r]) * 16 + col[r])   # their size and colour
            grid.sync()
            for i in range(g0, NP, gs):
                r = find(par, i)
                if size[i] > 0 and r != i:                             # a root that got linked
                    cuda.atomic.add(size, r, size[i])
                    size[i] = 0
                    for c3 in range(3):                                # momentum adds up
                        cuda.atomic.add(mom, (r, c3), mom[i, c3])
                        mom[i, c3] = 0.0
                        acc[i, c3] = 0.0
                    if froot[i]:
                        froot[r] = 1
                par[i] = r
            grid.sync()
            for i in range(g0, NP, gs):
                r = par[i]
                if r == i and cbest[i] >= 0:                           # a merged mass: the biggest one's
                    b = cbest[i]                                       # colour, or random if all were free
                    col[i] = (b & 15) if b >= 32 else int32(rnd(i + 18 * NP, t) % uint32(ncol))
                    cbest[i] = -1
                if froot[r] and fixed[i] == 0:                         # joined the seed's cluster
                    fixed[i] = 1
                    for c3 in range(3):                                # and stops
                        mom[i, c3] = 0.0
                        acc[i, c3] = 0.0
                    amass[i] = osize[i]
                    cuda.atomic.add(stats, 2, 1)
                if (size[r] >= 2 or froot[r]) and clus[i] == 0:
                    clus[i] = 1
                    act[i] = 2                                         # its contacts now count: re-check
                    btime[i] = t
                    cuda.atomic.add(stats, 1, 1)
                    cuda.atomic.max(stats, 0, t)
            if g0 == 0:
                flag[0] = 0
            grid.sync()
        R = misc[es]
        while R < NP:                                                  # G. the masses that fired, in turn
            m = size[R]
            colR = col[R]                                              # (its pieces keep it)
            if par[R] == R and fixed[R] == 0 and m >= cap and m >= 2:
                for i in range(g0, NP, gs):                            # G1. its centroid
                    if par[i] == R:
                        p = pos[i]
                        cuda.atomic.add(misc, 3, p & PM)
                        cuda.atomic.add(misc, 4, (p >> PB) & PM)
                        cuda.atomic.add(misc, 5, p >> (2 * PB))
                grid.sync()
                cx = misc[3] / m
                cy = misc[4] / m
                cz = misc[5] / m
                for it in range((NP + gs - 1) // gs):                  # G2. nearest other floating mass
                    i = g0 + it * gs                                   # (min over each warp first, then
                    v = int64(BIG)                                     #  one atomic per warp)
                    if i < NP:
                        r = par[i]
                        if r != R and fixed[i] == 0 and size[r] >= 2:
                            p = pos[i]
                            ddx = int64(p & PM) - int64(cx + 0.5)
                            ddy = int64((p >> PB) & PM) - int64(cy + 0.5)
                            ddz = int64(p >> (2 * PB)) - int64(cz + 0.5)
                            v = (ddx * ddx + ddy * ddy + ddz * ddz) * (1 << IB) + i
                    for d in (16, 8, 4, 2, 1):
                        u = cuda.shfl_xor_sync(0xFFFFFFFF, v, d)
                        if u < v:
                            v = u
                    if lane == 0 and v < BIG:
                        cuda.atomic.min(misc, 1, v)
                grid.sync()
                if misc[1] < BIG:                                      # away from it or towards it
                    jn = int32(misc[1] & ((1 << IB) - 1))
                    pj = pos[jn]
                    ex = cx - float(pj & PM)
                    ey = cy - float((pj >> PB) & PM)
                    ez = cz - float(pj >> (2 * PB))
                    if crule:                                          # other colour: towards it
                        tow = colour(col, R, ncol) != colour(col, par[jn], ncol)
                    else:                                              # aggregator: towards it
                        tow = is_agg(R, feat[colour(col, R, ncol), 3] if evo else pagg, NP)
                    if tow:
                        ex, ey, ez = -ex, -ey, -ez
                else:                                                  # no other mass: random direction
                    ex = rnd(R + 3 * NP, t) / 4294967296.0 - 0.5
                    ey = rnd(R + 4 * NP, t) / 4294967296.0 - 0.5
                    ez = rnd(R + 5 * NP, t) / 4294967296.0 - 0.5
                nrm = math.fabs(ex) + math.fabs(ey) + math.fabs(ez)    # L1-normalise: a speed of 1 is
                if nrm < 1e-9:                                         # one hop per step on the lattice
                    ex, ey, ez, nrm = 1.0, 0.0, 0.0, 1.0
                ex /= nrm
                ey /= nrm
                ez /= nrm
                ef = efrac
                vr = v0
                if evo:                                                # its colour's ejecta fraction, speed
                    cR = colour(col, R, ncol)
                    ef = scaled(efrac, feat[cR, 1])
                    ef = 0.01 if ef < 0.01 else (0.99 if ef > 0.99 else ef)
                    vr = scaled(v0, feat[cR, 2])
                    vr = 1.0 if vr > 1.0 else vr
                kk = int32(ef * m + 0.5)                           # ejecta size: efrac of the mass,
                kk = 1 if kk < 1 else (m - 1 if kk > m - 1 else kk)    # at least 1, leaving at least 1
                for i in range(g0, NP, gs):                            # G3. histogram of how far each
                    if par[i] == R:                                    #     member lies towards (ex,ey,ez)
                        p = pos[i]
                        sp = (float(p & PM) - cx) * ex + (float((p >> PB) & PM) - cy) * ey + (float(p >> (2 * PB)) - cz) * ez
                        q = int32((sp + hr) * hsc)
                        q = 0 if q < 0 else (HB - 1 if q >= HB else q)
                        ejc[i] = 0
                        cuda.atomic.add(hist, q, 1)
                        cuda.atomic.max(misc, 11, q)                   # highest occupied bin
                grid.sync()
                if g0 == 0:                                            # G4. threshold: the kk furthest ones
                    tot = 0
                    b = misc[11]                                       # (not HB - 1: far fewer reads)
                    while b > 0 and tot + hist[b] < kk:
                        tot += hist[b]
                        b -= 1
                    misc[6] = b
                    misc[7] = kk - tot
                grid.sync()
                tb = misc[6]
                for i in range(g0, NP, gs):                            # G5. pick the ejecta
                    if par[i] == R:
                        p = pos[i]
                        sp = (float(p & PM) - cx) * ex + (float((p >> PB) & PM) - cy) * ey + (float(p >> (2 * PB)) - cz) * ez
                        q = int32((sp + hr) * hsc)
                        q = 0 if q < 0 else (HB - 1 if q >= HB else q)
                        e = q > tb
                        if q == tb and cuda.atomic.add(misc, 8, 1) < misc[7]:
                            e = True
                        ejc[i] = 1 if e else 2                         # part label: 1 ejecta, 2 rest
                        if e:
                            cuda.atomic.add(misc, 2, 1)
                        hist[q] = 0
                grid.sync()
                ke = misc[2]                                           # ejecta size actually taken
                kr = m - ke
                px0 = mom[R, 0]                                        # (read by every thread before G7
                py0 = mom[R, 1]                                        #  changes it: syncs in between)
                pz0 = mom[R, 2]
                jk = vr * ke * kr / m                                  # impulse: relative speed vr
                vex = (px0 * ke / m + ex * jk) / ke                    # ejecta: along (ex, ey, ez)
                vey = (py0 * ke / m + ey * jk) / ke
                vez = (pz0 * ke / m + ez * jk) / ke
                vrx = (px0 * kr / m - ex * jk) / kr                    # the rest recoils the other way
                vry = (py0 * kr / m - ey * jk) / kr
                vrz = (pz0 * kr / m - ez * jk) / kr
                for i in range(g0, NP, gs):                            # G6. split: each part falls apart
                    if ejc[i]:                                         #     into the pieces that touch
                        par[i] = i                                     #     (26-neighbourhood)
                grid.sync()
                for i in range(g0, NP, gs):
                    if ejc[i] == 0:
                        continue
                    p = pos[i]
                    x = int32(p & PM)
                    y = int32((p >> PB) & PM)
                    z = int32(p >> (2 * PB))
                    for cz in range((z - 1 if z > 0 else 0) >> CS, ((z + 1 if z < L - 1 else z) >> CS) + 1):
                        for cy in range((y - 1 if y > 0 else 0) >> CS, ((y + 1 if y < L - 1 else y) >> CS) + 1):
                            for cx in range((x - 1 if x > 0 else 0) >> CS, ((x + 1 if x < L - 1 else x) >> CS) + 1):
                                c = (cz * NCC + cy) * NCC + cx
                                q0 = cstart[c]
                                for q in range(q0, q0 + ccount[c]):
                                    pj = spos[q]
                                    dx = int32(pj & PM) - x
                                    dy = int32((pj >> PB) & PM) - y
                                    dz = int32(pj >> (2 * PB)) - z
                                    if -1 <= dx <= 1 and -1 <= dy <= 1 and -1 <= dz <= 1:
                                        j = order[q]
                                        if j != i and ejc[j] == ejc[i]:
                                            union(par, i, j)
                grid.sync()
                for i in range(g0, NP, gs):
                    if ejc[i]:
                        par[i] = find(par, i)
                        size[i] = 0
                grid.sync()
                for i in range(g0, NP, gs):
                    if ejc[i]:
                        cuda.atomic.add(size, par[i], 1)
                grid.sync()
                for i in range(g0, NP, gs):                            # G7. pieces get their part's velocity
                    if ejc[i]:
                        r = par[i]
                        sz = size[r]
                        if sz < 2 and clus[i]:                         # a piece of one is a free particle
                            clus[i] = 0
                            btime[i] = 0
                            cuda.atomic.sub(stats, 1, 1)
                        if r == i:
                            e = ejc[i] == 1
                            mom[i, 0] = (vex if e else vrx) * sz
                            mom[i, 1] = (vey if e else vry) * sz
                            mom[i, 2] = (vez if e else vrz) * sz
                            for c3 in range(3):
                                acc[i, c3] = 0.0
                            froot[i] = 0
                            col[i] = colR
                            gid[i] = int32(t)                          # no piece of this ejection sticks to
                            gtil[i] = int32(t + grace)                 # another one during the grace period
                        else:
                            for c3 in range(3):
                                mom[i, c3] = 0.0
                                acc[i, c3] = 0.0
                        ejc[i] = 0                                     # (only i reads its own label here)
                if g0 == 0:
                    cuda.atomic.add(stats, 3, 1)
            grid.sync()                                                # (everyone has read misc)
            if g0 == 0:                                                # reset the ejection scratch
                misc[es] = NP
                misc[1] = BIG
                for c3 in range(2, 6):
                    misc[c3] = 0
                misc[8] = 0
                misc[11] = 0
            grid.sync()
            for i in range(g0, NP, gs):                                # the next mass that fired
                if fire[i] == t and i > R:
                    cuda.atomic.min(misc, es, i)
            grid.sync()
            R = misc[es]
        if evo:                                                        # H. births and endings of masses
            for i in range(g0, NP, gs):
                ism = par[i] == i and fixed[i] == 0 and size[i] >= 2
                b = born[i]
                if ism and b == 0:                                     # born: maybe mutate its colour
                    born[i] = t
                    c = colour(col, i, ncol)
                    if rnd(i + 9 * NP, t) * INV < evp[0] * math.exp(-surv[c] / evp[1]):
                        cuda.atomic.add(nmut, c, 1)
                        for f in range(NF):
                            if rnd(i + (10 + f) * NP, t) * INV < evp[2 + f]:
                                feat[c, f] = rnd(i + (10 + NF + f) * NP, t) * INV
                elif b != 0 and not ism:                               # ended: its lifetime counts
                    born[i] = 0
                    c = colour(col, i, ncol)
                    cuda.atomic.add(lsum, c, float(t - b))
                    cuda.atomic.add(lcnt, c, 1)


@cuda.jit
def pack(pos, par, size, fixed, btime, amass, col, out, info, N, t_last, anchor, hthr, ncol):
    """3 words per particle for the browser: the packed position; shade (0..254 = bound, by bind time;
    255 = free) | fixed << 8 (1 = part of the seed's cluster) | flags << 16 (1 = hidden: the switched-off
    anchor; 2 = in a floating mass of size >= hthr, with its colour (colour rule) in bits 2-4); and the
    attach mass, or the mass's root for flag 2 (its particles aren't fixed, so they have no attach mass).
    info: [mobile clusters, particles in mobile clusters, mobile clusters of each colour x KMAX]."""
    i = cuda.grid(1)
    NP = N + 1
    if i >= NP:
        return
    if i == N:
        sh = 0                                                         # the seed: bound first
    elif btime[i] > 0:
        sh = int32(254.0 * btime[i] / t_last)
    else:
        sh = 255
    fl = 1 if (i == N and anchor == 0) else 0
    aux = amass[i]
    r = par[i]
    if fixed[i] == 0 and size[r] >= 2:
        cuda.atomic.add(info, 1, 1)
        if r == i:
            cuda.atomic.add(info, 0, 1)
            cuda.atomic.add(info, 2 + colour(col, i, ncol), 1)
        if size[r] >= hthr:                                           # near the ceiling: coloured by root
            fl |= 2 | (colour(col, r, ncol) << 2)
            aux = r
    out[3 * i] = pos[i]
    out[3 * i + 1] = uint32(sh | (int32(fixed[i]) << 8) | (fl << 16))
    out[3 * i + 2] = uint32(aux)


class Sim:
    maxblocks = None          # most co-resident blocks for the cooperative launch, found on first run
    window = True             # the 3-in-2x2x2 binding rule
    cap = 200                 # mass ceiling: a floating mass this big ejects a piece (OFF = never)
    v0 = 0.5                  # ejection: relative speed of the two parts (voxels/step)
    erate = 0.01              # ejection: probability per step that a mass at the ceiling fires (first order)
    efrac = 0.25              # ejection: fraction of the mass ejected
    decay = False             # first-order decay: every floating mass fires at rate erate, whatever its size
    pagg = 0.5                # probability that a floating mass is an aggregator (else a diffuser)
    crule = False             # colour rule: eject away from a mass of the same colour, towards another colour
    ncol = 3                  # colours for the colour rule (and evolution)
    dref = 20.0               # colour rule: distance (voxels) at which a mass fires at the eject rate k
    evo = False               # evolution: each colour has its own features, mutating less the longer it survives
    mu = 0.2                  # evolution: chance that a birth mutates its colour, for a colour with no survival yet
    tau = 1000.0              # evolution: survival (steps) that cuts the mutation chance by a factor e
    wmut = (0.5, 0.5, 0.5, 0.5)   # evolution: chance a mutation redraws each feature (rate, ejecta, speed, aggregator)
    hfrac = 0.8               # viewer: masses at >= this fraction of the ceiling get their own colour
    damp = 0.002              # momentum damping per step

    def __init__(self, seed=0, L=200, N=10_000):
        self.L = L = int(min(max(L, 8), LMAX))
        self.N = N = int(min(max(N, 1), NMAX))
        self.NP = NP = N + 1                                           # + the immobile seed, particle N
        self.CS, _, NCELL = geometry(L)
        rng = np.random.default_rng(seed)
        x, y, z = (rng.integers(0, L, NP, dtype=np.int64) for _ in range(3))
        x[N] = y[N] = z[N] = L // 2
        fixed = np.zeros(NP, np.uint8)
        fixed[N] = 1
        d = cuda.to_device
        self.d_pos = d((x | (y << PB) | (z << (2 * PB))).astype(np.uint32))
        self.d_par = d(np.arange(NP, dtype=np.int32))
        self.d_size = d(np.ones(NP, np.int32))
        self.d_fixed = d(fixed)
        self.d_clus = d(fixed.copy())                                  # the seed counts as a cluster
        self.d_btime = d(np.zeros(NP, np.int64))
        self.d_blocked = d(np.zeros(NP, np.int32))
        self.d_act = d(np.zeros(NP, np.uint8))
        self.d_mv = cuda.device_array(NP, np.int8)
        self.d_froot = d(fixed.copy())                                 # 'fixed' flag per root
        self.d_key = cuda.device_array(NP, np.int32)
        self.d_slot = cuda.device_array(NP, np.int32)
        self.d_ccount = cuda.device_array(NCELL, np.int32)
        self.d_cstart = cuda.device_array(NCELL, np.int32)
        self.d_order = cuda.device_array(NP, np.int32)
        self.d_spos = cuda.device_array(NP, np.uint32)
        self.d_sroot = cuda.device_array(NP, np.int32)
        self.d_slot2 = cuda.device_array(NP, np.int32)
        self.d_ccount2 = cuda.device_array(NCELL, np.int32)
        self.d_cstart2 = cuda.device_array(NCELL, np.int32)
        self.d_order2 = cuda.device_array(NP, np.int32)
        self.d_spos2 = cuda.device_array(NP, np.uint32)
        self.d_sroot2 = cuda.device_array(NP, np.int32)
        self.d_osize = cuda.device_array(NP, np.int32)
        self.d_amass = d(np.zeros(NP, np.int32))
        self.d_mom = d(np.zeros((NP, 3), np.float32))
        self.d_acc = d(np.zeros((NP, 3), np.float32))
        self.d_gid = d(np.zeros(NP, np.int32))
        self.d_gtil = d(np.zeros(NP, np.int32))
        self.d_fire = d(np.zeros(NP, np.int32))
        misc = np.zeros(16, np.int64)
        misc[1] = BIG
        misc[12] = misc[13] = NP
        self.d_misc = d(misc)
        self.d_hist = d(np.zeros(HB, np.int32))
        self.d_ejc = d(np.zeros(NP, np.uint8))
        self.d_flag = d(np.zeros(5, np.int32))
        self.d_stats = d(np.zeros(4, np.int64))
        self.d_born = d(np.zeros(NP, np.int32))
        self.d_feat = d(rng.random((KMAX, NF)).astype(np.float32))      # each colour's random features
        self.d_surv = d(np.zeros(KMAX, np.float64))
        self.d_lsum = d(np.zeros(KMAX, np.float64))
        self.d_lcnt = d(np.zeros(KMAX, np.int32))
        self.d_nmut = d(np.zeros(KMAX, np.int32))
        self.d_evp = d(np.zeros(2 + NF, np.float32))
        self.evo_was = False
        self.d_col = d(rng.integers(0, KMAX, NP).astype(np.int8))       # colour per root
        self.d_cbest = d(np.full(NP, -1, np.int64))
        self.d_csum = d(np.zeros((NP, 3), np.int64))
        self.d_srcs = cuda.device_array(NP, np.int32)
        self.d_cand = cuda.device_array(NP, np.int32)
        self.rng = rng
        self.d_out = cuda.device_array(3 * NP, np.uint32)
        self.d_info = cuda.device_array(2 + KMAX, np.int32)
        self.t = 0
        self.chunk = 10                                                # steps per launch (adapts)
        self.anchor = True                                             # central anchor present

    def _launch(self, blocks, n):
        grace = int(min(2000, 6 / max(self.v0, 0.003)))              # long enough to separate by ~6 voxels
        cap = 2 if self.decay else int(min(self.cap, OFF))
        if self.evo:
            if not self.evo_was:                                       # births are tracked from now on
                self.d_born[:] = 0
                self.d_lsum[:] = 0
                self.d_lcnt[:] = 0
            self.d_evp.copy_to_device(np.array([self.mu, max(self.tau, 1.0), *self.wmut], np.float32))
        self.evo_was = self.evo
        advance[blocks, TPB](self.d_pos, self.d_par, self.d_size, self.d_fixed, self.d_clus, self.d_btime,
                             self.d_blocked, self.d_act, self.d_mv, self.d_froot, self.d_key, self.d_slot,
                             self.d_ccount, self.d_cstart, self.d_order, self.d_spos,
                             self.d_sroot, self.d_slot2, self.d_ccount2, self.d_cstart2, self.d_order2,
                             self.d_spos2, self.d_sroot2,
                             self.d_osize, self.d_amass,
                             self.d_mom, self.d_acc, self.d_gid, self.d_gtil, self.d_misc, self.d_hist, self.d_ejc,
                             self.d_fire, self.d_flag, self.d_stats,
                             self.d_born, self.d_feat, self.d_surv, self.d_lsum, self.d_lcnt, self.d_nmut, self.d_evp,
                             self.d_col, self.d_cbest, self.d_csum, self.d_srcs, self.d_cand,
                             self.L, self.N, self.CS, self.t, n,
                             int(self.window), cap, float(self.v0), float(self.damp), grace, int(self.anchor),
                             float(self.erate), float(self.efrac), float(self.pagg), int(self.crule),
                             int(self.ncol), int(self.evo), float(self.dref))

    def run(self, steps, budget=None):
        """Run up to `steps` steps; with a budget (seconds), stop early once it is used up. Each launch is
        timed, and the steps per launch adapt so a launch takes about LAUNCH seconds."""
        if Sim.maxblocks is None:                                      # compile, then size the grid
            self._launch(1, 0)
            kernel = next(iter(advance.overloads.values()))
            Sim.maxblocks = kernel.max_cooperative_grid_blocks(TPB)
        blocks = max(1, min(Sim.maxblocks, (self.NP + TPB - 1) // TPB))
        t_start = time.perf_counter()
        while steps > 0:
            n = min(steps, self.chunk)
            t_launch = time.perf_counter()
            self._launch(blocks, n)
            cuda.synchronize()
            now = time.perf_counter()
            self.chunk = max(1, min(500, int(n * LAUNCH / max(now - t_launch, 1e-4))))
            self.t += n
            steps -= n
            if budget is not None and now - t_start > budget:
                break

    def set_anchor(self, on):
        """Switch the central anchor on or off (between kernel launches, on the host).

        Off: the anchor is hidden and ignored, and the system that was stuck to it is set floating:
        it is regrouped by contact (26-neighbourhood), each piece becoming a mobile cluster at rest
        (a piece of one particle is free again) with a random colour. On: the anchor is back at the centre and checks its
        neighbourhood on the next step, so anything already touching it sticks."""
        if on == self.anchor:
            return
        self.anchor = on
        N = self.N
        act = self.d_act.copy_to_host()
        if on:
            act[N] = 2
            self.d_act = cuda.to_device(act)
            return
        par, size = self.d_par.copy_to_host(), self.d_size.copy_to_host()
        fixed, froot = self.d_fixed.copy_to_host(), self.d_froot.copy_to_host()
        clus, btime = self.d_clus.copy_to_host(), self.d_btime.copy_to_host()
        amass, gid = self.d_amass.copy_to_host(), self.d_gid.copy_to_host()
        mom, acc, stats = self.d_mom.copy_to_host(), self.d_acc.copy_to_host(), self.d_stats.copy_to_host()
        pos = self.d_pos.copy_to_host().astype(np.int64)
        col = self.d_col.copy_to_host()
        members = [int(i) for i in np.nonzero(par == par[N])[0] if i != N]
        where = {}
        for i in members:
            where.setdefault((pos[i] & PM, (pos[i] >> PB) & PM, pos[i] >> (2 * PB)), []).append(i)
        seen = set()
        for start in members:                                          # connected pieces (26-neighbourhood)
            if start in seen:
                continue
            piece, stack = [], [start]
            seen.add(start)
            while stack:
                i = stack.pop()
                piece.append(i)
                x, y, z = pos[i] & PM, (pos[i] >> PB) & PM, pos[i] >> (2 * PB)
                for dx in (-1, 0, 1):
                    for dy in (-1, 0, 1):
                        for dz in (-1, 0, 1):
                            for j in where.get((x + dx, y + dy, z + dz), ()):
                                if j not in seen:
                                    seen.add(j)
                                    stack.append(j)
            r = min(piece)
            par[piece] = r
            size[piece] = 0
            size[r] = len(piece)
            col[r] = self.rng.integers(0, self.ncol)
            if len(piece) == 1:                                        # a lone particle is free again
                clus[piece] = 0
                btime[piece] = 0
        idx = np.array(members, dtype=np.int64)
        if len(idx):
            fixed[idx] = froot[idx] = amass[idx] = gid[idx] = 0
            mom[idx] = acc[idx] = 0.0
            act[idx] = 2
        par[N], size[N] = N, 1                                          # the anchor stays put, alone
        stats[1] = int(clus[:N].sum())
        stats[2] = int(fixed[:N].sum())
        for name, arr in (('par', par), ('size', size), ('fixed', fixed), ('froot', froot), ('clus', clus),
                          ('btime', btime), ('amass', amass), ('gid', gid), ('mom', mom), ('acc', acc),
                          ('stats', stats), ('act', act), ('col', col)):
            setattr(self, 'd_' + name, cuda.to_device(arr))

    def frame(self):
        """Pack all particles for the viewer (12 bytes each); also return counts for the info line."""
        t_last, nbound, nfixed, nejected = self.d_stats.copy_to_host()
        self.d_info[:] = 0
        hthr = self.NP + 1 if self.cap >= OFF else max(2, math.ceil(self.hfrac * self.cap))   # (ceiling off: none)
        pack[(self.NP + 255) // 256, 256](self.d_pos, self.d_par, self.d_size, self.d_fixed, self.d_btime,
                                          self.d_amass, self.d_col, self.d_out, self.d_info, self.N, max(float(t_last), 1.0),
                                          int(self.anchor), hthr, int(self.ncol))
        cnt = self.d_info.copy_to_host()
        feat, surv, nmut = self.d_feat.copy_to_host(), self.d_surv.copy_to_host(), self.d_nmut.copy_to_host()
        sc = lambda base, u: float(base * SPREAD ** (2 * u - 1) if self.evo else base)   # (evolution off: sliders)
        colours = [dict(k=sc(self.erate, f[0]), efrac=min(0.99, max(0.01, sc(self.efrac, f[1]))),
                        v0=min(1.0, sc(self.v0, f[2])), agg=float(f[3] if self.evo else self.pagg),
                        surv=float(s), mut=int(m), n=int(c))
                   for f, s, m, c in zip(feat[:self.ncol], surv, nmut, cnt[2:])]
        info = dict(t=self.t, L=self.L, N=self.N, free=self.N - int(nbound), mobile=int(cnt[1]),
                    clusters=int(cnt[0]), fixed=int(nfixed), ejections=int(nejected), colours=colours)
        return self.d_out.copy_to_host().tobytes(), info


Sim().run(1); Sim().frame()                                         # compile kernels up front
print(f'kernels compiled ({Sim.maxblocks} co-resident blocks x {TPB} threads)')


In [ ]:
# The viewer is a plain web page served from this kernel (no Jupyter widgets, which VS Code fails to load here).
# It polls /frame for the packed particles and sends control changes to /ctl.
import gc
PAGE = r"""<!doctype html><html><head><meta charset="utf-8"><style>
  body { margin: 0; background: #2b2118; color: #eee; font: 13px system-ui, sans-serif; }
  #bar { display: flex; gap: 10px; align-items: center; padding: 8px; flex-wrap: wrap; }
  button { font: inherit; padding: 3px 10px; }
  input[type=number] { font: inherit; width: 84px; }
  canvas { display: block; cursor: grab; touch-action: none; }
  #info { padding: 0 8px 6px; font-variant-numeric: tabular-nums; }
  .row { display: flex; gap: 10px; align-items: center; padding: 0 8px 8px; flex-wrap: wrap; }
  .tog { min-width: 84px; }  .tog.on { background: #2f6bff; color: #fff; border: 1px solid #2f6bff; }
  .dim { opacity: .7; }
  #evotab table { border-collapse: collapse; font-variant-numeric: tabular-nums; }
  #evotab td, #evotab th { padding: 1px 8px; text-align: right; }
  .sw { display: inline-block; width: 12px; height: 12px; border-radius: 2px; vertical-align: middle; }
</style></head><body>
<div id="bar">
  steps/frame <input id="speed" type="range" min="0" max="4.5" step="0.1" value="2"> <span id="sv">100</span>
  <button id="pause">Pause</button> <button id="reset">Reset</button> <button id="stop">Stop</button>
  &nbsp; box <input id="Lsz" type="number" min="8" max="1000" step="1" value="200"> voxels a side,
  <input id="Nsz" type="number" min="1" max="1000000" step="1000" value="10000"> particles
  <span class="dim">(applied on Reset)</span>
  <span class="dim">drag = rotate, scroll = zoom, double-click = reset view</span>
</div>
<div class="row">
  mass ceiling <input id="cap" type="range" min="0.5" max="6.5" step="0.05" value="2.3"> <span id="capv">200</span>
  &nbsp; eject speed <input id="v0" type="range" min="0.05" max="1" step="0.05" value="0.5"> <span id="vv">0.50</span>
  &nbsp; eject rate <input id="erate" type="range" min="-4" max="0" step="0.1" value="-2"> <span id="erv">0.010 /step</span>
  &nbsp; ejecta <input id="efrac" type="range" min="1" max="99" step="1" value="25"> <span id="efv">25%</span>
  <button id="decay" class="tog">First-order decay: off</button>
  &nbsp; damping <input id="damp" type="range" min="-4" max="-1" step="0.1" value="-2.7"> <span id="mv">0.0020</span>
</div>
<div class="row">
  aggregators <input id="pagg" type="range" min="0" max="100" step="1" value="50"> <span id="pav">50%</span>
  <span class="dim">(eject towards the nearest mass; the rest are diffusers, ejecting away from it)</span>
  &nbsp; <button id="crule" class="tog">Colour rule: off</button>
  colours <input id="ncol" type="range" min="2" max="8" step="1" value="3"> <span id="ncv">3</span>
  &nbsp; rate &prop; 1/distance: k at <input id="dref" type="range" min="0" max="2.5" step="0.05" value="1.3"> <span id="drv">20 voxels</span>
  <span class="dim">(same colour: eject away, other colour: eject towards, at a rate inversely proportional to the distance to the
  nearest mass; replaces the aggregator rule)</span>
</div>
<div class="row">
  <button id="evo" class="tog">Evolution: off</button>
  mutation chance <input id="mu" type="range" min="0" max="100" step="1" value="20"> <span id="muv">20%</span>
  &nbsp; survival scale <input id="tau" type="range" min="1" max="5" step="0.1" value="3"> <span id="tauv">1,000 steps</span>
  &nbsp; mutation weights: rate <input id="w0" class="w" type="range" min="0" max="100" step="1" value="50"> <span id="w0v">50%</span>
  ejecta <input id="w1" class="w" type="range" min="0" max="100" step="1" value="50"> <span id="w1v">50%</span>
  speed <input id="w2" class="w" type="range" min="0" max="100" step="1" value="50"> <span id="w2v">50%</span>
  aggregator <input id="w3" class="w" type="range" min="0" max="100" step="1" value="50"> <span id="w3v">50%</span>
</div>
<div id="evotab" class="row"></div>
<div class="row">
  <button id="hl" class="tog on">Ceiling colours: on</button>
  colour masses &ge; <input id="hfrac" type="range" min="0" max="100" step="1" value="80"> <span id="hfv">80%</span> of the ceiling
  &nbsp; others' opacity <input id="dim" type="range" min="0" max="100" step="1" value="30"> <span id="dmv">30%</span>
  &nbsp; <button id="anchor" class="tog on">Anchor: on</button>
  <button id="blink" class="tog on">Blinks: on</button>
  delay for a single particle <input id="delay" type="range" min="1.7" max="4" step="0.05" value="3"> <span id="dv">1,000 ms</span>
  &nbsp; <label><input id="win" type="checkbox" checked> 3-in-2x2x2 binding</label>
</div>
<div id="info">waiting for the simulation...</div>
<canvas id="cv"></canvas>
<script>
// Drawing: every particle is splatted into a pixel buffer with a depth buffer, so there is no per-frame
// sort and a million particles stay interactive. Opaque particles go first and write depth; translucent
// ones (free particles, and with ceiling colours on everything outside the coloured masses) are then
// blended over whatever they are in front of.
const S = 600, OFF = 1 << 30;
const cv = document.getElementById('cv'), dpr = window.devicePixelRatio || 1, W = Math.round(S * dpr);
cv.width = cv.height = W; cv.style.width = cv.style.height = S + 'px';
const ctx = cv.getContext('2d');
const img = ctx.createImageData(W, W), pix = new Uint32Array(img.data.buffer), zb = new Float32Array(W * W);
const rgb = (r, g, b) => ((255 << 24) | (b << 16) | (g << 8) | r) >>> 0;              // (little-endian RGBA)
const BGC = rgb(0xCC, 0x77, 0x22), WHITE = rgb(255, 255, 255), BLUEC = rgb(0x2f, 0x6b, 0xff);
const GREY = Array.from({ length: 255 }, (_, g) => rgb(g, g, g));
const PALRGB = [[230, 25, 75], [60, 180, 75], [67, 99, 216], [255, 225, 25], [240, 50, 230], [66, 212, 244],
                [145, 30, 180], [250, 190, 212]];
const PAL = PALRGB.map(c => rgb(...c));                             // colour rule / evolution: K colours
function hsl(hd, s, l) {                                            // -> packed colour
  const f = k => { const a = s * Math.min(l, 1 - l), m = (k + hd / 30) % 12; return Math.round(255 * (l - a * Math.max(-1, Math.min(m - 3, 9 - m, 1)))); };
  return rgb(f(0), f(8), f(4));
}
const mcol = new Map();                                             // a random colour per root, from a hash
const massColour = r => { let c = mcol.get(r); if (c === undefined) { c = hsl((Math.imul(r + 1, 2654435761) >>> 0) % 360, 0.85, 0.55); mcol.set(r, c); } return c; };
function blend(c0, c1, a) {
  const b = 1 - a;
  return rgb(((c0 & 255) * b + (c1 & 255) * a) | 0, (((c0 >>> 8) & 255) * b + ((c1 >>> 8) & 255) * a) | 0,
             (((c0 >>> 16) & 255) * b + ((c1 >>> 16) & 255) * a) | 0);
}

let L = 200, n = 0, words = new Uint32Array(0), fid = -1, synced = false, az = 0.6, tilt = 0.4, zoom = 1, drag = null, queued = false, paused = false;
let capOn = true, hlOn = true, cruleOn = false, evoOn = false, DIM = 0.3;

function draw() {
  queued = false;
  const h = (L - 1) / 2, sc = zoom * W / (L * 1.8), now = performance.now(), c2 = W / 2;  // device px per voxel
  const ca = Math.cos(az), sa = Math.sin(az), ce = Math.cos(tilt), se = Math.sin(tilt);
  const proj = (x, y, z) => {                                       // z is up; az turns about z
    const dx = x - h, dy = y - h, dz = z - h, x1 = ca * dx - sa * dy, d1 = -(sa * dx + ca * dy);
    return [c2 + x1 * sc, c2 - (ce * dz - se * d1) * sc];
  };
  pix.fill(BGC); zb.fill(-Infinity);
  const fs = Math.max(1, Math.round(Math.max(1.5 * dpr, 0.7 * sc))), bs = Math.max(2, Math.round(Math.max(2.5 * dpr, 1.1 * sc)));
  const hlAct = hlOn && capOn, palOn = cruleOn || evoOn, sysOn = blinkOn && now >= seedBlink && now < seedBlink + BLINK, anyBlink = blinkOn && blinkAt.size > 0;
  for (let pass = 0; pass < 2; pass++) {                            // 0: opaque, 1: translucent
    for (let i = 0; i < n; i++) {
      const w1 = words[3 * i + 1], f = w1 >>> 16;
      if (f & 1) continue;                                          // the switched-off anchor
      const s = w1 & 255, hl = hlAct && (f & 2);
      let a = s === 255 ? 0.5 : 1;                                  // free particles: faint
      if (hlAct && !hl) a *= DIM;                                   // outside the coloured masses: dimmed
      if ((a < 1) !== (pass === 1) || a <= 0) continue;
      const p = words[3 * i], dx = (p & 1023) - h, dy = ((p >>> 10) & 1023) - h, dz = (p >>> 20) - h;
      const x1 = ca * dx - sa * dy, d1 = -(sa * dx + ca * dy), depth = se * dz + ce * d1;   // larger = nearer
      const sx = c2 + x1 * sc, sy = c2 - (ce * dz - se * d1) * sc;
      const b = anyBlink ? blinkAt.get(i) : undefined;
      const blink = b !== undefined ? now >= b && now < b + BLINK  // joinee: after DELAY / mass
                                    : sysOn && ((w1 >>> 8) & 255) === 1;   // the seed's system: at once
      const col = blink ? BLUEC : hl ? (palOn ? PAL[(f >>> 2) & 7] : massColour(words[3 * i + 2]))
                                     : s === 255 ? WHITE : GREY[s];          // black early -> white late
      const sz = s === 255 ? fs : bs, x0 = Math.round(sx - sz / 2), y0 = Math.round(sy - sz / 2);
      const xa = x0 < 0 ? 0 : x0, xb = x0 + sz > W ? W : x0 + sz, ya = y0 < 0 ? 0 : y0, yb = y0 + sz > W ? W : y0 + sz;
      for (let yy = ya; yy < yb; yy++) {
        for (let o = yy * W + xa, oe = yy * W + xb; o < oe; o++) {
          if (depth > zb[o]) {
            if (pass === 0) { pix[o] = col; zb[o] = depth; } else pix[o] = blend(pix[o], col, a);
          }
        }
      }
    }
  }
  ctx.putImageData(img, 0, 0);
  ctx.strokeStyle = 'rgba(60,30,0,0.45)'; ctx.lineWidth = dpr; ctx.beginPath();           // cube outline
  for (const [a, b] of [[0,1],[2,3],[4,5],[6,7],[0,2],[1,3],[4,6],[5,7],[0,4],[1,5],[2,6],[3,7]]) {
    const pa = proj(a & 1 ? L - 1 : 0, a & 2 ? L - 1 : 0, a & 4 ? L - 1 : 0);
    const pb = proj(b & 1 ? L - 1 : 0, b & 2 ? L - 1 : 0, b & 4 ? L - 1 : 0);
    ctx.moveTo(pa[0], pa[1]); ctx.lineTo(pb[0], pb[1]);
  }
  ctx.stroke();
  for (const [i, b] of blinkAt) if (now >= b + BLINK) blinkAt.delete(i);
  if (blinkOn && (now < seedBlink + BLINK || blinkAt.size)) schedule();   // keep animating while blinking
}
const schedule = () => { if (!queued) { queued = true; requestAnimationFrame(draw); } };

// Blinks: when a floating particle or system joins the seed's cluster, every particle already in that
// cluster turns blue at once, and the arriving particles turn blue after DELAY / mass (mass = size of the
// floating system they came in); each pixel stays blue for BLINK ms and then goes back to its colour.
const BLINK = 250;                                                  // blink duration (ms)
let DELAY = 1000, blinkOn = true, seedBlink = -1e9, blinkAt = new Map();
function onFrame(nw) {
  const now = performance.now();
  if (blinkOn && words.length === nw.length) {                      // same run: look for new attachments
    let any = false;
    for (let i = 0, m = nw.length / 3; i < m; i++) {
      if (((nw[3 * i + 1] >>> 8) & 255) && !((words[3 * i + 1] >>> 8) & 255)) {
        blinkAt.set(i, now + DELAY / Math.max(1, nw[3 * i + 2]));
        any = true;
      }
    }
    if (any) seedBlink = now;
  } else if (words.length !== nw.length) { blinkAt.clear(); mcol.clear(); }
  words = nw; n = nw.length / 3;
}

async function poll() {
  try {
    const r = await fetch('frame?have=' + fid, { cache: 'no-store' });
    const st = JSON.parse(r.headers.get('X-State')), body = await r.arrayBuffer();
    if (st.t === undefined) { setTimeout(poll, 100); return; }                          // no frame yet
    if (!synced) {                                                  // show the simulation's real settings
      synced = true;
      win.checked = st.window;
      setTog(anchorB, st.anchor, 'Anchor'); setTog(decayB, st.decay, 'First-order decay'); setTog(cruleB, st.crule, 'Colour rule');
      cruleOn = st.crule;
      Lsz.value = st.L; Nsz.value = st.N;
      capS.value = st.cap >= OFF ? capS.max : Math.log10(st.cap); showCap(st.cap);
      v0S.value = st.v0; document.getElementById('vv').textContent = st.v0.toFixed(2);
      erateS.value = Math.log10(st.erate); showErate(st.erate);
      efracS.value = Math.round(100 * st.efrac); showEfrac(st.efrac);
      paggS.value = Math.round(100 * st.pagg); showPct('pav', st.pagg);
      ncolS.value = st.ncol; document.getElementById('ncv').textContent = st.ncol;
      drefS.value = Math.log10(st.dref); showDref(st.dref);
      hfracS.value = Math.round(100 * st.hfrac); showPct('hfv', st.hfrac);
      setTog(evoB, st.evo, 'Evolution'); evoOn = st.evo;
      muS.value = Math.round(100 * st.mu); showPct('muv', st.mu);
      tauS.value = Math.log10(st.tau); showTau(st.tau);
      st.wmut.forEach((w, f) => { wS[f].value = Math.round(100 * w); showPct('w' + f + 'v', w); });
      dampS.value = Math.log10(st.damp); document.getElementById('mv').textContent = st.damp.toFixed(4);
      speed.value = Math.log10(st.speed); document.getElementById('sv').textContent = st.speed;
    }
    if (body.byteLength) { L = st.L; onFrame(new Uint32Array(body)); fid = st.fid; capOn = st.cap < OFF; schedule(); showColours(st); }   // redraw only on new data
    document.getElementById('info').textContent = st.error ? 'simulation stopped: ' + st.error :
      `box ${st.L}³, ${st.N.toLocaleString()} particles   step ${st.t.toLocaleString()}   free ${st.free.toLocaleString()}   ` +
      `mobile clusters ${st.clusters.toLocaleString()} (${st.mobile.toLocaleString()} particles)   ` +
      `stuck to seed ${st.fixed.toLocaleString()}   ejections ${st.ejections.toLocaleString()}` +
      (st.alive ? '' : '   [stopped]');
  } catch (e) { document.getElementById('info').textContent = 'lost connection to the kernel (rerun the cell)'; }
  setTimeout(poll, 33);
}
function showColours(st) {                                          // evolution: each colour's features
  const el = document.getElementById('evotab');
  if (!evoOn && !cruleOn) { el.innerHTML = ''; return; }
  const rows = st.colours.map((c, i) => `<tr><td><span class="sw" style="background:rgb(${PALRGB[i]})"></span></td>` +
    `<td>${c.n.toLocaleString()}</td><td>${c.k.toPrecision(2)}</td><td>${Math.round(100 * c.efrac)}%</td>` +
    `<td>${c.v0.toFixed(2)}</td><td>${Math.round(100 * c.agg)}%</td><td>${Math.round(c.surv).toLocaleString()}</td>` +
    `<td>${c.mut.toLocaleString()}</td></tr>`).join('');
  el.innerHTML = '<table><tr><th></th><th>masses</th><th>eject rate</th><th>ejecta</th><th>speed</th><th>aggregator</th>' +
    '<th>mean survival (steps)</th><th>mutations</th></tr>' + rows + '</table>' +
    (evoOn ? '' : '<span class="dim">(evolution off: the sliders apply to every colour)</span>');
}
const ctl = q => fetch('ctl?' + q);
const speed = document.getElementById('speed');
speed.oninput = () => { const n = Math.round(10 ** speed.value); document.getElementById('sv').textContent = n; ctl('speed=' + n); };
document.getElementById('pause').onclick = e => { paused = !paused; e.target.textContent = paused ? 'Resume' : 'Pause'; ctl('pause=' + (+paused)); };
const Lsz = document.getElementById('Lsz'), Nsz = document.getElementById('Nsz');
document.getElementById('reset').onclick = () => {                 // a new run, with the size settings
  Lsz.value = Math.max(8, Math.min(1000, Math.round(+Lsz.value || 200)));
  Nsz.value = Math.max(1, Math.min(1000000, Math.round(+Nsz.value || 10000)));
  ctl('L=' + Lsz.value + '&N=' + Nsz.value);
};
const delay = document.getElementById('delay');
delay.oninput = () => { DELAY = Math.round(10 ** delay.value); document.getElementById('dv').textContent = DELAY.toLocaleString() + ' ms'; };
const win = document.getElementById('win');
win.onchange = () => ctl('window=' + (+win.checked));
const setTog = (b, on, label) => { b.classList.toggle('on', on); b.textContent = label + ': ' + (on ? 'on' : 'off'); };
const isOn = b => b.classList.contains('on');
const anchorB = document.getElementById('anchor');
anchorB.onclick = () => { setTog(anchorB, !isOn(anchorB), 'Anchor'); ctl('anchor=' + (+isOn(anchorB))); };
const blinkB = document.getElementById('blink');
blinkB.onclick = () => { blinkOn = !isOn(blinkB); setTog(blinkB, blinkOn, 'Blinks'); if (!blinkOn) blinkAt.clear(); schedule(); };
const decayB = document.getElementById('decay');
decayB.onclick = () => { setTog(decayB, !isOn(decayB), 'First-order decay'); ctl('decay=' + (+isOn(decayB))); };
const cruleB = document.getElementById('crule'), ncolS = document.getElementById('ncol');
cruleB.onclick = () => { cruleOn = !isOn(cruleB); setTog(cruleB, cruleOn, 'Colour rule'); schedule(); ctl('crule=' + (+cruleOn)); };
ncolS.oninput = () => { document.getElementById('ncv').textContent = ncolS.value; ctl('ncol=' + ncolS.value); };
const drefS = document.getElementById('dref');
const showDref = v => { document.getElementById('drv').textContent = (v < 10 ? v.toFixed(1) : Math.round(v)) + ' voxels'; };
drefS.oninput = () => { const v = 10 ** drefS.value; showDref(v); ctl('dref=' + v); };
const capS = document.getElementById('cap'), v0S = document.getElementById('v0'), dampS = document.getElementById('damp');
const showCap = n => { document.getElementById('capv').textContent = n >= OFF ? 'off' : n.toLocaleString(); };
capS.oninput = () => { const n = +capS.value >= +capS.max ? OFF : Math.max(2, Math.round(10 ** capS.value)); showCap(n); ctl('cap=' + n); };
v0S.oninput = () => { document.getElementById('vv').textContent = (+v0S.value).toFixed(2); ctl('v0=' + v0S.value); };
const erateS = document.getElementById('erate'), efracS = document.getElementById('efrac');
const showErate = k => { document.getElementById('erv').textContent = k >= 0.999 ? '1 /step (instant)' : k.toPrecision(2) + ' /step'; };
const showEfrac = f => { document.getElementById('efv').textContent = Math.round(100 * f) + '%'; };
erateS.oninput = () => { const k = 10 ** erateS.value; showErate(k); ctl('erate=' + k); };
efracS.oninput = () => { const f = efracS.value / 100; showEfrac(f); ctl('efrac=' + f); };
const paggS = document.getElementById('pagg'), hfracS = document.getElementById('hfrac'), dimS = document.getElementById('dim');
const showPct = (id, f) => { document.getElementById(id).textContent = Math.round(100 * f) + '%'; };
paggS.oninput = () => { const f = paggS.value / 100; showPct('pav', f); ctl('pagg=' + f); };
hfracS.oninput = () => { const f = hfracS.value / 100; showPct('hfv', f); ctl('hfrac=' + f); };
dimS.oninput = () => { DIM = dimS.value / 100; showPct('dmv', DIM); schedule(); };
const evoB = document.getElementById('evo'), muS = document.getElementById('mu'), tauS = document.getElementById('tau');
const wS = [0, 1, 2, 3].map(f => document.getElementById('w' + f));
const showTau = v => { document.getElementById('tauv').textContent = Math.round(v).toLocaleString() + ' steps'; };
evoB.onclick = () => { evoOn = !isOn(evoB); setTog(evoB, evoOn, 'Evolution'); schedule(); ctl('evo=' + (+evoOn)); };
muS.oninput = () => { const f = muS.value / 100; showPct('muv', f); ctl('mu=' + f); };
tauS.oninput = () => { const v = 10 ** tauS.value; showTau(v); ctl('tau=' + v); };
wS.forEach((s, f) => { s.oninput = () => { const w = s.value / 100; showPct('w' + f + 'v', w); ctl('w' + f + '=' + w); }; });
const hlB = document.getElementById('hl');
hlB.onclick = () => { hlOn = !isOn(hlB); setTog(hlB, hlOn, 'Ceiling colours'); schedule(); };
dampS.oninput = () => { const d = 10 ** dampS.value; document.getElementById('mv').textContent = d.toFixed(4); ctl('damp=' + d); };
document.getElementById('stop').onclick = () => ctl('stop=1');

cv.addEventListener('pointerdown', e => { drag = [e.clientX, e.clientY]; cv.setPointerCapture(e.pointerId); cv.style.cursor = 'grabbing'; });
cv.addEventListener('pointermove', e => {
  if (!drag) return;
  az += (e.clientX - drag[0]) * 0.01;
  tilt = Math.max(-1.55, Math.min(1.55, tilt + (e.clientY - drag[1]) * 0.01));
  drag = [e.clientX, e.clientY]; schedule();
});
const end = () => { drag = null; cv.style.cursor = 'grab'; };
cv.addEventListener('pointerup', end); cv.addEventListener('pointercancel', end);
cv.addEventListener('wheel', e => { e.preventDefault(); zoom = Math.max(0.3, Math.min(10, zoom * Math.exp(-e.deltaY * 0.0015))); schedule(); }, { passive: false });
cv.addEventListener('dblclick', () => { az = 0.6; tilt = 0.4; zoom = 1; schedule(); });
schedule(); poll();
</script></body></html>"""

if 'server' in globals():                        # rerunning the cell: stop the previous sim + server
    state['alive'] = False
    server.shutdown()
    server.server_close()
for th in threading.enumerate():                 # and wait for every earlier sim thread to end, so two never
    if th.name == 'sim_loop' or th.name.endswith('(sim_loop)'):   # share the GPU (one started by an older
        th.join(timeout=30)                      # version of this cell reads the global `state`, so it would
                                                 # otherwise pick up the new one below and run on)

state = dict(alive=True, speed=100, pause=False, reset=False, error=None, frame=(b'', {}), fid=0, want=True,
             L=200, N=10_000, window=Sim.window, cap=Sim.cap, v0=Sim.v0, erate=Sim.erate, efrac=Sim.efrac,
             decay=Sim.decay, damp=Sim.damp, anchor=True, pagg=Sim.pagg, crule=Sim.crule, ncol=Sim.ncol,
             hfrac=Sim.hfrac, dref=Sim.dref, evo=Sim.evo, mu=Sim.mu, tau=Sim.tau, wmut=list(Sim.wmut), repack=False)

def sim_loop(state):
    """All CUDA work happens on this one thread; the web server only reads state['frame']. A frame is
    packed only once the browser has the previous one (state['want']), so big systems don't pack and
    copy frames nobody will see. It gets its own `state`, so a rerun of the cell can't revive it."""
    sim = Sim(L=state['L'], N=state['N'])
    dirty = True
    try:
        while state['alive']:
            t_start = time.perf_counter()
            if state['reset']:
                state['reset'], sim = False, None              # free the old arrays before the new ones
                gc.collect()
                cuda.current_context().deallocations.clear()
                sim = Sim(seed=int(time.time()) % 10_000, L=state['L'], N=state['N'])
                dirty = True
            sim.window, sim.cap, sim.v0, sim.damp = state['window'], state['cap'], state['v0'], state['damp']
            sim.erate, sim.efrac, sim.decay = state['erate'], state['efrac'], state['decay']
            sim.pagg, sim.crule, sim.ncol, sim.hfrac = state['pagg'], state['crule'], state['ncol'], state['hfrac']
            sim.dref = state['dref']
            sim.evo, sim.mu, sim.tau, sim.wmut = state['evo'], state['mu'], state['tau'], tuple(state['wmut'])
            sim.set_anchor(state['anchor'])
            if not state['pause']:
                sim.run(state['speed'], budget=0.5)   # (a big system stops early, so the controls stay live)
                dirty = True
            if state['repack']:                  # a new colour threshold shows even while paused
                state['repack'], dirty = False, True
            if dirty and state['want']:          # fid tells the browser whether there is anything new
                state['want'], dirty = False, False
                state['frame'], state['fid'] = sim.frame(), state['fid'] + 1
            time.sleep(max(0.0, 1 / 30 - (time.perf_counter() - t_start)))   # ~30 updates/s, no idle gap
    except Exception as e:
        state['error'] = f'{type(e).__name__}: {e}'
        state['alive'] = False

class Handler(BaseHTTPRequestHandler):
    def log_message(self, *args):                # keep the notebook output quiet
        pass

    def do_GET(self):
        url = urlparse(self.path)
        extra = {}
        if url.path == '/':
            body, ctype = PAGE.encode(), 'text/html; charset=utf-8'
        elif url.path == '/frame':
            fid, (data, info) = state['fid'], state['frame']
            have = parse_qs(url.query).get('have', [''])[0]
            if have == str(fid):                 # the browser is up to date: pack the next frame
                state['want'] = True
            body, ctype = (b'' if have == str(fid) else data), 'application/octet-stream'   # empty = unchanged
            extra['X-State'] = json.dumps(dict(info, fid=fid, alive=state['alive'], error=state['error'],
                                               speed=state['speed'], window=state['window'],
                                               cap=state['cap'], v0=state['v0'], erate=state['erate'],
                                               efrac=state['efrac'], decay=state['decay'], damp=state['damp'],
                                               anchor=state['anchor'], pagg=state['pagg'], crule=state['crule'],
                                               ncol=state['ncol'], hfrac=state['hfrac'], dref=state['dref'],
                                               evo=state['evo'],
                                               mu=state['mu'], tau=state['tau'], wmut=state['wmut']))
        elif url.path == '/ctl':
            q = {k: v[0] for k, v in parse_qs(url.query).items()}
            if 'speed' in q:
                state['speed'] = max(1, int(q['speed']))
            if 'pause' in q:
                state['pause'] = q['pause'] == '1'
            if 'reset' in q:
                state['reset'] = True
            if 'L' in q:
                state['L'] = min(LMAX, max(8, int(q['L'])))
                state['reset'] = True
            if 'N' in q:
                state['N'] = min(NMAX, max(1, int(q['N'])))
                state['reset'] = True
            if 'window' in q:
                state['window'] = q['window'] == '1'
            if 'anchor' in q:
                state['anchor'] = q['anchor'] == '1'
            if 'cap' in q:
                state['cap'] = min(OFF, max(2, int(q['cap'])))
            if 'v0' in q:
                state['v0'] = min(1.0, max(0.0, float(q['v0'])))
            if 'erate' in q:
                state['erate'] = min(1.0, max(0.0, float(q['erate'])))
            if 'efrac' in q:
                state['efrac'] = min(0.99, max(0.01, float(q['efrac'])))
            if 'decay' in q:
                state['decay'] = q['decay'] == '1'
            if 'damp' in q:
                state['damp'] = min(1.0, max(0.0, float(q['damp'])))
            if 'pagg' in q:
                state['pagg'] = min(1.0, max(0.0, float(q['pagg'])))
            if 'crule' in q:
                state['crule'] = q['crule'] == '1'
            if 'dref' in q:
                state['dref'] = min(1e4, max(1.0, float(q['dref'])))
            if 'ncol' in q:
                state['ncol'] = min(8, max(2, int(q['ncol'])))
                state['repack'] = True
            if 'hfrac' in q:
                state['hfrac'] = min(1.0, max(0.0, float(q['hfrac'])))
                state['repack'] = True
            if 'evo' in q:
                state['evo'] = q['evo'] == '1'
            if 'mu' in q:
                state['mu'] = min(1.0, max(0.0, float(q['mu'])))
            if 'tau' in q:
                state['tau'] = min(1e6, max(1.0, float(q['tau'])))
            for f in range(NF):
                if f'w{f}' in q:
                    state['wmut'][f] = min(1.0, max(0.0, float(q[f'w{f}'])))
            if 'stop' in q:
                state['alive'] = False
            body, ctype = b'ok', 'text/plain'
        else:
            self.send_error(404)
            return
        self.send_response(200)
        self.send_header('Content-Type', ctype)
        self.send_header('Content-Length', str(len(body)))
        self.send_header('Cache-Control', 'no-store')
        for k, v in extra.items():
            self.send_header(k, v)
        self.end_headers()
        self.wfile.write(body)

class Server(ThreadingHTTPServer):
    allow_reuse_address = False                  # on Windows, reuse would let a 2nd kernel silently share the port

PORT = 8765
try:
    server = Server(('127.0.0.1', PORT), Handler)
except OSError:                                  # port taken (e.g. another kernel): use any free port
    server = Server(('127.0.0.1', 0), Handler)
threading.Thread(target=server.serve_forever, daemon=True).start()
threading.Thread(target=sim_loop, args=(state,), name='sim_loop', daemon=True).start()
url = f'http://127.0.0.1:{server.server_port}/'
display(HTML(f'Viewer: <a href="{url}" target="_blank">{url}</a> (open in a browser if the frame below stays blank)'))
display(IFrame(url, width=640, height=800))
